# Brain-as-LLM Local Linearization Experiment

**Goal:** test whether human intracranial neural population dynamics can be treated, locally, like hidden-state dynamics in an LLM:

$$\hat{x}_{t+\Delta} = A\,x_t + b + \hat{w}_t$$

| LLM | Brain |
|---|---|
| hidden state $h[l,t]$ | population state $x[l,t]$ |
| layer $l$ | anatomical module (temporal cortex → entorhinal → hippocampus) |
| hidden-state dimension | electrode **channel** (never "neuron") |
| token index | 20-ms time bin |

**Story:** locally linearizable → finite temporal prediction horizon → degradation under state shift (placebo → scopolamine OOD).

Full specification: [ANALYSIS_SPEC.md](ANALYSIS_SPEC.md)

**Guardrails (from spec):**
- Fit dynamics **within patient** only; aggregate statistics across patients (patient = inferential unit).
- Split by **whole trials**, never individual time points; all preprocessing/hyperparameters from training data only.
- Residual model $\hat{w}_t = g(x_t)$ trained on **training residuals** — never add back test residuals.
- OOD evaluation uses the **frozen** model (same $A$, $b$, $g$, normalization).
- Report negative results honestly (spec §28).

## 0. Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Paths (DATA_DIR to be pointed at the provided data folder) ----
HERE = Path.cwd() if (Path.cwd() / "ANALYSIS_SPEC.md").exists() else Path("NeuroAnalysis")
DATA_DIR = HERE / "data"          # <- point to the provided data folder
RESULTS_DIR = HERE / "results"
PRED_DIR = HERE / "predictions"
FIG_DIR = HERE / "figures"
for d in (RESULTS_DIR, PRED_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Signal parameters (spec §2, §4, §5) ----
FS = 1000                          # native sampling rate (Hz)
EPOCH_MS = (-500, 1500)            # baseline: -500..0, encoding: 0..1500
BANDS = {
    "slow_theta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 16),
    "beta": (16, 32),
    "gamma": (32, 64),
}
PRIMARY_BAND = "theta"
TOKEN_STRIDE_MS = 20               # brain "token" spacing
CAUSAL_WINDOW_MS = 40              # x(t) = mean power over [t-40ms, t] (no future leakage)

# ---- Horizons (spec §6) ----
PRIMARY_HORIZON_MS = 20
HORIZONS_MS = [10, 20, 50, 100, 200]

# ---- Splits (spec §8): by whole trials ----
SPLIT_FRACS = (0.6, 0.2, 0.2)      # train / val / test
RANDOM_SEED = 0

# ---- Anatomical layers (spec §3) ----
LAYERS = ["temporal_cortex", "entorhinal", "hippocampus"]

# ---- Figure colors (spec §23): identical across all panels ----
COLOR_TRUE = "black"
COLOR_LINEAR = "#1f4ea1"          # blue: A x_t
COLOR_RESIDUAL = "#c1272d"        # red:  A x_t + w_hat
COLOR_ID = "#7f9fbf"              # muted: placebo / ID
COLOR_OOD = "#bf8f7f"             # muted: scopolamine / OOD

rng = np.random.default_rng(RANDOM_SEED)
print(f"DATA_DIR = {DATA_DIR.resolve()}  (exists: {DATA_DIR.exists()})")

## 1. Load & validate data (STEP 1)

Load electrode localization + trial metadata. Validate:
- patients, sessions, trial counts per condition (placebo / scopolamine; FR / AR)
- electrode counts per region per patient (simultaneous coverage map)
- epoch alignment (-500..+1500 ms @ 1000 Hz)

*Awaiting data folder — loader will be implemented against the actual file format.*

In [ ]:
# TODO(data): implement loader once the data folder + paper are provided.
# Expected products:
#   electrode_metadata: DataFrame [patient, channel, region, x/y/z or atlas label]
#   trial_metadata:     DataFrame [patient, session, trial_id, task, drug_condition, onset]
#   raw_epochs:         dict[patient] -> array (n_trials, n_channels, n_samples)

## 2. Preprocessing → band-power state (STEPS 2-3)

Per electrode & band: baseline-normalized Hilbert power; causal token features
$x(t) = \text{mean power over } [t-40\,\text{ms}, t]$ at 20-ms stride → per-patient, per-region state matrices `(n_trials, n_tokens, n_channels)`.

In [ ]:
# TODO(data): bandpass (per BANDS) -> Hilbert envelope -> baseline normalization (-500..0 ms,
# training-set statistics) -> causal 20-ms tokenization.
# Document: zero-phase filtering is acausal; causal-processing sensitivity analysis if practical (spec §5).

## 3. Trial-wise splits (STEP 4)

Group by `trial_id` (session-aware if sessions exist): 60/20/20 train/val/test. No time-point-level splitting.

In [ ]:
# TODO: grouped trial split + repeated group CV when trial count permits.

## 4. Models & baselines (STEPS 5-6, 8)

- **Persistence:** $\hat{x}_{t+\Delta} = x_t$
- **Mean trajectory:** $\hat{x}_{t+\Delta} = \mu(t+\Delta)$ (training condition mean)
- **Diagonal AR** (optional): each channel predicts itself
- **Ridge linear:** $\hat{x}_{t+\Delta} = A x_t + b$, $\lambda$ chosen on training/validation only
- **Residual model:** $\hat{w}_t = g(x_t)$, low-capacity MLP $N \to \min(32, 2N) \to N$ (or kernel ridge for small $N$), trained on *training residuals*, early stopping on validation

Metrics (spec §11-12): population $R^2$ (primary), NRMSE, Pearson r, cosine similarity, relative residual energy, $\Delta R^2$.

In [ ]:
# TODO: model fitting + metric functions (pure functions taking (train, val, test) state arrays).

## 5. Proof of concept (spec §27)

One patient (sufficient hippocampal electrodes) · hippocampus · theta 4-8 Hz · 0-1500 ms · stride 20 ms · Δ = 20 ms.

Outputs: true-vs-predicted trajectory; $R^2$ for persistence / $Ax$ / $Ax+\hat{w}$; residual-energy ratio; placebo-ID vs scopolamine-OOD comparison.

**Gate:** full batch analysis only runs after this passes sanity checks.

In [ ]:
# TODO: end-to-end POC.

## 6. Horizon sweep (STEP 7, Analysis 3)

Δ ∈ {10, 20, 50, 100, 200} ms × {persistence, $Ax$, $Ax+\hat{w}$} × patients → `results/horizon_metrics.csv`. No monotonicity forced.

In [ ]:
# TODO: horizon sweep.

## 7. OOD: drug-state shift (STEPS 10-11, Analyses 4 + control)

Train placebo → frozen model → test held-out placebo (ID) vs scopolamine (OOD). Then the symmetric reverse (train scopolamine). Report $R^2_{ID} - R^2_{OOD}$ and residual-energy change → `results/ood_metrics.csv`.

In [ ]:
# TODO: OOD analysis (frozen model, both directions). Secondary: FR<->AR task transfer (spec §16).

## 8. Robustness & controls (STEPS 12, 14; spec §20)

Frequency bands (all 5) · temporal shuffle (should collapse) · trial shuffle (should strongly decrease) · electrode-count subsampling · patient-wise leave-one-out on the central claim.

In [ ]:
# TODO: shuffle controls + band robustness.

## 9. Cross-region "layer" mapping (STEP 13, Analysis 5)

Temporal cortex → entorhinal → hippocampus; lags δ ∈ {0, 10, 20, 50, 100} ms (selected on training data, all reported). Compare recurrent-only $R_l x^{(l+1)}_t$ vs source+recurrent $A_l x^{(l)}_t + R_l x^{(l+1)}_t$ → `results/layer_metrics.csv`.

In [ ]:
# TODO: cross-region models (only where simultaneous coverage exists; never fabricate layers).

## 10. Matrix diagnostics (spec §18)

Eigenvalue spectrum (complex plane), spectral radius, singular values, effective rank, condition number of each fitted $A$.

In [ ]:
# TODO: A-matrix diagnostics.

## 11. Patient-level statistics (STEP 15, spec §19)

One summary metric per patient per comparison → paired statistics across patients (linear vs linear+residual; ID vs OOD). Individual patient dots in all plots.

In [ ]:
# TODO: paired stats (Wilcoxon / paired t as appropriate).

## 12. Main four-panel figure (STEP 16, spec §22-24)

A: brain-as-network mapping · B: local linearization + representative real prediction (patient closest to median improvement — predefined, no cherry-picking) · C: horizon curves · D: OOD degradation. Colors: black = truth, blue = $Ax$, red = $Ax+\hat{w}$, identical across panels. Vector output.

In [ ]:
# TODO: figure assembly -> figures/main_figure.pdf/.png